In [1]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [7]:
from langgraph.graph import StateGraph,END,START
from agents.nodes.context import contextAdder
from agents.nodes.router import routerNode
from agents.subgraph.analysisgraph import buildAnalysisGraph
from agents.subgraph.tradinggraph import buildTradingGraph
from agents.subgraph.engagementgraph import buildEngagementGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from agents.state import AgentState
from config.settings import settings
from config.constants import(
    INTENT_MATCH_INSIGHT,
    INTENT_BET_ADVICE,
    INTENT_STOCK_PREDICT,
    INTENT_SMART_ALERT,
    INTENT_CHAT
)
import json,logging
logger=logging.getLogger(__name__)
_agent=None

In [3]:
async def outputChecker(state):
    output=state.get("output",{})
    if not output or state.get("intent")==INTENT_CHAT:
        return{
            "quality-score":1.0
        }
    llm=ChatGoogleGenerativeAI(
        model=settings.GEMINI_MODEL,
        google_api_key=settings.GEMINI_API_KEY,
        temperature=0
    )
    prompt=ChatPromptTemplate.from_messages([
        ("system",
         "Score the quality of this AI-generated output from 0.0 to 1.0. "
         "Consider: completeness, accuracy of reasoning, actionability, specificity. "
         "Respond with ONLY a JSON: {{\"score\": 0.85, \"feedback\": \"brief reason\"}}"),
        ("human","Query: {query}\n\nOutput:\n{output}"),
    ])
    try:
        result=await (prompt|llm).ainvoke({
            "query":state.get("query",""),
            "output":json.dumps(output,default=str)[:1000]
        })
        content=result.content.strip()
        if "```" in content:
            content=content.split("```")[1].split("```")[0]
            if content.startswith("json"):
                content=content[4:]
        parsed=json.loads(content)
        score=parsed.get("score",0.7)
    except Exception:
        score=0.7
    return{
        "quality-score":score,
        "reflection-count":state.get("reflection-count",0)+1
    }

In [4]:
async def addMetadata(state):
    output=state.get("output",{})
    output["_meta"]={
        "intent":state.get("intent",""),
        "confidence":state.get("confidence",0),
        "qualityScore":state.get("quality-score",0),
        "reflections":state.get("reflection-count",0),
        "cost":state.get("cost",0),
        "tokens":state.get("tokens",{})
    }
    return{
        "output":output
    }

In [5]:
def routeAfterRouter(state):
    intent=state.get("intent","")
    if intent in (INTENT_MATCH_INSIGHT,INTENT_SMART_ALERT):
        return "analysisGraph"
    elif intent in (INTENT_BET_ADVICE,INTENT_STOCK_PREDICT):
        return "tradingGraph"
    else:
        return "engagementGraph"

In [6]:
def shouldReflect(state):
    intent=state.get("intent","")
    if intent==INTENT_CHAT:
        return"addMetadata"
    score=state.get("quality-score",1.0)
    count=state.get("reflection-count",0)
    if score<0.7 and count<2:
        return routeAfterRouter(state)
    return "addMetadata"

In [8]:
def buildSupervisor():
    graph=StateGraph(AgentState)
    graph.add_node("contextAdder",contextAdder)
    graph.add_node("router",routerNode)
    graph.add_node("analysisGraph",buildAnalysisGraph())
    graph.add_node("tradingGraph",buildTradingGraph())
    graph.add_node("engagementGraph",buildEngagementGraph())
    graph.add_node("outputChecker",outputChecker)
    graph.add_node("addMetadata",addMetadata)

    graph.add_edge(START,"contextAdder")
    graph.add_edge("contextAdder","router")
    graph.add_conditional_edges("router",routeAfterRouter,{
        "analysisGraph":"analysisGraph",
        "tradingGraph":"tradingGraph",
        "engagementGraph":"engagementGraph"
    })
    graph.add_edge("analysisGraph","outputChecker")
    graph.add_edge("tradingGraph","outputChecker")
    graph.add_edge("engagementGraph","outputChecker")
    graph.add_conditional_edges("outputChecker",shouldReflect,{
        "analysisGraph":"analysisGraph",
        "tradingGraph":"tradingGraph",
        "engagementGraph":"engagementGraph",
        "addMetadata":"addMetadata"
    })
    graph.add_edge("addMetadata",END)
    return graph.compile()

In [9]:
def getAgent():
    global _agent
    if _agent is None:
        _agent=buildSupervisor()
        logger.info("agent supervisor built")
    return _agent